# NVV-Pipeline Experiment - Full-GT and Part-GT
- yml: environment.yml
- env: nvv_isolation_pipeline
- last updated: 28.03.2026


## Setup
- Imports
- Config

In [ ]:
#Setup
import os
import sys
from pathlib import Path

project_root = Path(os.getcwd()).parent
sys.path.append(str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

from config.load_config import load_config
from config.path_factory import print_paths

from evaluation.analysis_loader import load_and_compare_workspaces

from evaluation.analysis_tables import build_rq1_capability_tables

from evaluation.analysis_tables import (
    build_setting_summary_table,
    build_best_all_screened_values_matrix,
    build_derivative_matrix,
    build_top_k_runs_table
)
from evaluation.analysis_plots import (
    plot_setting_overview,
    plot_derivative_group_comparison,
    plot_top_k_runs_per_setting,
)

cfg_path_nvs_full_gt = project_root / "experiments" / "pipeline_evaluation" / "config_nvs38k_full_gt.yaml"
cfg_path_nvs_part_gt = project_root / "experiments" / "pipeline_evaluation" / "config_nvs38k_part_gt.yaml"
#cfg_path_vocals_full_gt = project_root / "experiments" / "pipeline_evaluation" / "config_vocal_full_gt.yaml"
cfg_path_vocals_part_gt = project_root / "experiments" / "pipeline_evaluation" / "config_vocal_part_gt.yaml"

config = load_config(cfg_path_nvs_full_gt)
print_paths(cfg_path_nvs_full_gt)

## Collect Pipeline Evaluation Experiment Results 
- Experiment specs for comparison


In [ ]:
# specification of full run evaluation experiments to load and compare (all settings)
specs = [
    {
        "label": "nvs-38k_EN | full_gt",
        "cfg_path": cfg_path_nvs_full_gt,
        "dataset_name": "nvs38k_EN",
        "mode": "full_gt",
    },
    {
        "label": "nvs-38k_EN | part_gt",
        "cfg_path": cfg_path_nvs_part_gt,
        "dataset_name": "nvs38k_EN",
        "mode": "part_gt",
    },
    {
        "label": "VOCAL RA1| part_gt",
        "cfg_path": cfg_path_vocals_part_gt,
        "dataset_name": "VOCAL_RA1",
        "mode": "part_gt",
    },
    {
        "label": "VOCAL RA2 | part_gt",
        "cfg_path": cfg_path_vocals_part_gt,
        "dataset_name": "VOCAL_RA2",
        "mode": "part_gt",
    },
]

analysis_bundle = load_and_compare_workspaces(
    specs=specs,
    param_names=[],
    param_pairs=[],
    )


views = analysis_bundle["combined_views"]

## RQ1 Pipeline Capability

In [ ]:

# RQ1 Pipeline Capability
#toDo: update artifact: insertion rate needed, remove precision and f1 for part_gt 

# --- collect RQ1 tables from the loaded settings ---
rq1_all = pd.concat(
    [
        bundle["results"]["rq1"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq1_tables = build_rq1_capability_tables(rq1_all)

print("RQ1 Capability Table (Full-GT):")
display(rq1_tables["full_gt"])

print("RQ1 Capability Table (Part-GT):")
display(rq1_tables["part_gt"])




## RQ2 Ranking Configuration: Single best

In [ ]:
from evaluation.analysis_tables import build_rq2a_single_ranking_tables

rq2a_single_all = pd.concat(
    [
        bundle["results"]["rq2a_single"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq2a_single_tables = build_rq2a_single_ranking_tables(
    rq2a_single_all,
    top_k=10,
)

print("RQ2a Single Ranking Tables:")
for setting_name, df_table in rq2a_single_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)

In [ ]:
rq2a_single_all[[
    "setting",
    "macro_mean_dice_eos_recall",
    "macro_mean_mean_dice_eos_tp",
]].head(20)

## RQ2 Configuration Combination: Selected Set

In [ ]:
from evaluation.analysis_tables import build_rq2a_selected_set_tables

rq2a_selected_all = pd.concat(
    [
        bundle["results"]["rq2a_selected_set"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq2a_selected_tables = build_rq2a_selected_set_tables(
    rq2a_single_all,
    rq2a_selected_all,
)

print("RQ2a Selected Set Tables:")
for setting_name, df_table in rq2a_selected_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)

## RQ2 Audio Derivative Aggregation

In [ ]:
# RQ2 Audio Derivative Aggregation - Just for inspection, not for final presentation. ToDo: update artifact with insertion rate and remove precision and f1 for part_gt
# Keep derivative matrices separated by metric / mode
derivative_matrix_f1 = build_derivative_matrix(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "full_gt"
    ],
    value_col="macro_mean_f1", #toDo: also recall and dice
)

derivative_matrix_recall = build_derivative_matrix(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "part_gt"
    ],
    value_col="macro_mean_recall", #toDo also eos and eos_tp and insertion_rate
)

print("\nDerivative Matrix (F1, full_gt):")
display(derivative_matrix_f1)
print("\nDerivative Matrix (Recall, part_gt):")
display(derivative_matrix_recall)



In [ ]:
# RQ2b Audio Derivative Aggregation - Group Comparison (Thesis Tables)
# toDo: insert EOS Recall to result table (RQ2b needs macro_mean_dice_eos_recall)
from evaluation.analysis_tables import build_rq2b_derivative_tables
## RQ2b Audio Derivative Aggregation

rq2b_tables = build_rq2b_derivative_tables(views["derivative_comparison"])

print("RQ2b Audio Derivative Table (Full-GT):")
display(rq2b_tables["full_gt"])

print("RQ2b Audio Derivative Table (Part-GT):")
display(rq2b_tables["part_gt"])

## RQ3 NVV Coverage

In [ ]:
## RQ3 NVV Coverage

from evaluation.analysis_tables import (
    build_rq3_full_gt_tables,
    build_rq3_part_gt_tables,
)
## RQ3 NVV Coverage

rq3_all = pd.concat(
    [
        bundle["results"]["rq3"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq3_full_gt_tables = build_rq3_full_gt_tables(rq3_all)
rq3_part_gt_tables = build_rq3_part_gt_tables(rq3_all)

print("RQ3 Coverage Tables (Full-GT):")
for setting_name, df_table in rq3_full_gt_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)

print("RQ3 Event Tables (Part-GT):")
for setting_name, df_table in rq3_part_gt_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)